In [1]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate


In [2]:
PLOT_FOLDER = "SLURMTHICKNESS_Verify"
target      = "interfacial_thickness"
SEED        = 655552
TEST_ROWS   = None
HPO_FOLDER  = "SLURMTHICKNESS"   # folder containing decoded_best_config.joblib


In [3]:
# Parameters
PLOT_FOLDER = "/gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMTHICKNESS_Verify"
HPO_FOLDER = "/gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMTHICKNESS"
TEST_ROWS = None
SEED = 655552


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn",
)

try:
    import tabpfn
    from tabpfn import TabPFNRegressor

    os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"
    print(f"TabPFN version: {tabpfn.__version__}")

except ImportError as exc:
    raise ImportError("tabpfn is not installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")


TabPFN version: 8.0.3
Thread limit set to 16 (SLURM_CPUS_PER_TASK)


In [5]:
df = pd.read_csv("../../DATASET_A4/interfacial_results_dataset_A4.csv")
print(f"Number of rows: {len(df)}")

if isinstance(TEST_ROWS, str) and TEST_ROWS.strip().lower() in ("", "none", "null"):
    TEST_ROWS = None
if TEST_ROWS is not None:
    TEST_ROWS = int(TEST_ROWS)
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())


Number of rows: 19361
Full mode: using all rows
Total samples: 19361

interfacial_thickness statistics:
count    19361.000000
mean         1.353666
std          0.682266
min          0.682573
25%          0.887208
50%          1.109909
75%          1.574425
max          5.659519
Name: interfacial_thickness, dtype: float64


In [6]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split — identical seed to HPO run ensures same split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")


Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   13552
Testing samples:    2904
Validation samples: 2905


In [7]:
# Load the optimized hyperparameters saved by the HPO run
config_path = os.path.join(HPO_FOLDER, f"TabPFN_{target}_decoded_best_config.joblib")
model_params = joblib.load(config_path)
print(f"Loaded config from: {config_path}")
print("\nModel parameters:")
for k, v in model_params.items():
    print(f"  {k}: {v}")

# Reconstruct TabPFNRegressor with the exact HPO-optimised params
model = TabPFNRegressor(**model_params)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)
y_val_pred   = model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="nm", y_val=y_val, y_val_pred=y_val_pred)


Loaded config from: /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMTHICKNESS/TabPFN_interfacial_thickness_decoded_best_config.joblib

Model parameters:
  average_before_softmax: True
  ignore_pretraining_limits: True
  model_path: /home/draju/.cache/tabpfn/tabpfn-v2.5-regressor-v2.5_small-samples.ckpt
  n_estimators: 4
  softmax_temperature: 0.8
  inference_config: {'FINGERPRINT_FEATURE': False, 'MIN_UNIQUE_FOR_NUMERICAL_FEATURES': 1, 'OUTLIER_REMOVAL_STD': None, 'POLYNOMIAL_FEATURES': 'no', 'PREPROCESS_TRANSFORMS': ({'append_original': True, 'categorical_name': 'ordinal_very_common_categories_shuffled', 'global_transformer_name': 'svd_quarter_components', 'name': 'none'},), 'REGRESSION_Y_PREPROCESS_TRANSFORMS': ('safepower',)}


Model Performance for interfacial_thickness

Training Set:
  R²:   0.999929
  RMSE: 0.005754 nm
  MAE:  0.000846 nm

Test Set:
  R²:   0.999890
  RMSE: 0.007220 nm
  MAE:  0.001065 nm

Validation Set:
  R²:   0.999908
  RMSE: 0.006463 nm
  MAE:  0.001009 nm


In [8]:
# Compare against the HPO run metrics
hpo_metrics_path = os.path.join(HPO_FOLDER, f"TabPFN_{target}_metrics.json")
with open(hpo_metrics_path) as f:
    hpo_metrics = json.load(f)

print("\n" + "=" * 60)
print("Performance comparison: HPO tuned vs Verify (loaded config)")
print("=" * 60)
print(f"  {"Metric":<20} {"HPO run":>12} {"Verify":>12}")
print("-" * 46)
for key in ["test_r2", "test_rmse", "test_mae", "val_r2", "val_rmse", "val_mae"]:
    hpo_val = hpo_metrics.get(key, float("nan"))
    ver_val = metrics.get(key, float("nan"))
    match = "✓" if abs(float(hpo_val) - float(ver_val)) < 1e-4 else "Δ"
    print(f"  {key:<20} {float(hpo_val):>12.6f} {float(ver_val):>12.6f}  {match}")
print("=" * 60)



Performance comparison: HPO tuned vs Verify (loaded config)
  Metric                    HPO run       Verify
----------------------------------------------
  test_r2                  0.999895     0.999890  ✓
  test_rmse                0.007064     0.007220  Δ
  test_mae                 0.001019     0.001065  ✓
  val_r2                   0.999905     0.999908  ✓
  val_rmse                 0.006557     0.006463  ✓
  val_mae                  0.001016     0.001009  ✓


In [9]:
cv_results = cross_validate(
    model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=1,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std()*2:.6f})")
print(f"Cross-Validation RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std()*2:.6f})")
print(f"Cross-Validation MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std()*2:.6f})")

# Compare CV with HPO run
print("\nCV comparison with HPO run:")
for key, scores, hpo_key in [
    ("R²",   cv_r2_scores,   "cv_r2_mean"),
    ("RMSE", cv_rmse_scores, "cv_rmse_mean"),
    ("MAE",  cv_mae_scores,  "cv_mae_mean"),
]:
    hpo_val = hpo_metrics.get(hpo_key, float("nan"))
    print(f"  CV {key:<6} HPO={float(hpo_val):.6f}  Verify={scores.mean():.6f}")


Cross-Validation R²:   0.998718 (+/- 0.001698)
Cross-Validation RMSE: 0.022793 (+/- 0.018342)
Cross-Validation MAE:  0.005430 (+/- 0.003934)

CV comparison with HPO run:
  CV R²     HPO=0.999246  Verify=0.998718
  CV RMSE   HPO=0.018420  Verify=0.022793
  CV MAE    HPO=0.004959  Verify=0.005430


In [10]:
os.makedirs(PLOT_FOLDER, exist_ok=True)

results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred, y_test_pred, y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
results_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"), index=False)

joblib.dump(model, os.path.join(PLOT_FOLDER, f"TabPFN_{target}_model.joblib"))

metrics.update({
    "cv_r2_scores":  cv_r2_scores.tolist(),
    "cv_r2_mean":    float(cv_r2_scores.mean()),
    "cv_r2_std":     float(cv_r2_scores.std()),
    "cv_rmse_scores":cv_rmse_scores.tolist(),
    "cv_rmse_mean":  float(cv_rmse_scores.mean()),
    "cv_rmse_std":   float(cv_rmse_scores.std()),
    "cv_mae_scores": cv_mae_scores.tolist(),
    "cv_mae_mean":   float(cv_mae_scores.mean()),
    "cv_mae_std":    float(cv_mae_scores.std()),
    "model":         "TabPFN_HPO_Verify",
    "features":      features,
    "target":        target,
    "seed":          SEED,
    "hpo_config":    str(config_path),
})

metrics_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Results saved to {PLOT_FOLDER}/")


Results saved to /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMTHICKNESS_Verify/


In [11]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")


Total notebook runtime: 0.45 minutes
